# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. Usaremos Wine Quality como caso didáctico de clasificación
multiclase: predeciremos la calidad original y no debe interpretarse como garantía comercial.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos siete runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.joblib` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==2.2.3" "matplotlib==3.11.1" "joblib==1.5.3" "xgboost==3.0.5"

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import joblib
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# -> Configurar identidad del lote y tracking de MLflow en Databricks
# ~> Alias de equipo no personal; adaptalo a tu pareja sin usar nombre, correo o matricula
STUDENT_ALIAS = "equipo-01"
# ~> Validar el alias: 3-24 caracteres, minusculas, numeros, _ o -
assert re.fullmatch(r"[a-z0-9_-]{3,24}", STUDENT_ALIAS), (
    "Usa 3-24 caracteres: minusculas, numeros, _ o -, sin datos personales."
)

BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE - 1.0) < 1e-9
MIN_VALIDATION_F1_MACRO = 0.28
QUALITY_MIN = 3
QUALITY_MAX = 8
NUM_CLASSES = QUALITY_MAX - QUALITY_MIN + 1
# Esta ruta es relativa a semana1/modules/01-mlflow-databricks-foundations/notebooks/.
DATASET_PATH = "../../../data/raw/WineQT.csv"
# ~> Allowlist de tipos confiables para skops (necesaria para XGBoost en MLflow 3.14)
SKOPS_TRUSTED_TYPES = [
    "numpy.dtype",
    "sklearn.tree._tree.Tree",
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBClassifier",
]

# > Configurar el tracking URI de Databricks para MLflow
mlflow.set_tracking_uri("databricks")

# >> Imprimir versiones y configuracion activa
print(
    f"MLflow={mlflow.__version__} · sklearn={sklearn.__version__} · "
    f"xgboost={xgboost.__version__} · "
    f"alias={STUDENT_ALIAS} · batch={BATCH_ID}"
)

## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío, falta el target o `Id` se cuela como predictor. Construye un `quality_report`
serializable como JSON y confirma que el target contiene al menos dos clases.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "quality"
EXPECTED_FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

# -> Validar el contrato de datos y crear X, y, quality_report
# ~> Verificar que todas las columnas obligatorias estan presentes
missing_columns = sorted(set(EXPECTED_FEATURES + [TARGET]) - set(data.columns))
assert not missing_columns, f"Faltan columnas obligatorias: {missing_columns}"

# ~> Verificar que el dataset no esta vacio y el target no tiene nulos
assert len(data) > 0, "El dataset esta vacio."
assert data[TARGET].notna().all(), "El target no puede contener valores nulos."
assert data[TARGET].nunique() >= 2, "El target debe contener al menos dos clases."
assert "Id" in data.columns, "WineQT debe incluir la columna tecnica Id."

# > Separar features (X) excluyendo Id, y mapear target a clases consecutivas desde 0
X = data[EXPECTED_FEATURES].copy()
quality_values = sorted(data[TARGET].astype(int).unique().tolist())
assert quality_values == list(range(QUALITY_MIN, QUALITY_MAX + 1)), (
    f"Se esperaban calidades consecutivas {QUALITY_MIN}..{QUALITY_MAX}; recibidas {quality_values}."
)
# ~> XGBoost exige clases consecutivas desde cero; el mapeo se conserva como artefacto
y = data[TARGET].astype(int) - QUALITY_MIN

# > Construir el quality_report serializable como JSON
quality_report = {
    "rows": int(len(data)),
    "columns": int(data.shape[1]),
    "duplicate_rows": int(data.duplicated().sum()),
    "target_distribution_quality": {str(k): int(v) for k, v in data[TARGET].value_counts().sort_index().items()},
    "target_distribution_encoded": {str(k): int(v) for k, v in y.value_counts().sort_index().items()},
    "missing_by_column": {k: int(v) for k, v in X.isna().sum().items()},
    "checks": {
        "required_columns": "passed",
        "non_empty": "passed",
        "multiclass_target": "passed",
        "quality_values": quality_values,
        "target_encoding": f"quality_class = {TARGET} - {QUALITY_MIN}",
        "id_excluded_from_features": "passed",
    },
}

# >> Mostrar tipos y nulos por columna
display(pd.DataFrame({"dtype": X.dtypes.astype(str), "missing": X.isna().sum()}))

# >> Imprimir el quality_report
print(json.dumps(quality_report, indent=2, ensure_ascii=False))

## 3. Divide 70 % train, 15 % validación y 15 % test, siempre estratificado

In [0]:
# -> Dividir datos en train (70%), validacion (15%) y test (15%) estratificado
# ~> Primera division: separar test (15%) del resto (85%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# ~> Segunda division: separar validacion del train
# ~> test_size = VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE) = 0.15 / 0.85
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE),
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

# > Verificar la estratificacion de train/validation sin inspeccionar etiquetas de test
split_report = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "quality_class_mean": [y_train.mean(), y_valid.mean()],
    },
    index=["train", "validation"],
)

# >> Mostrar el reporte del split
display(split_report)
print(f"Test reservado: {len(X_test)} filas; etiquetas y metricas ocultas hasta cerrar el gate.")
display(pd.DataFrame({
    "train": y_train.value_counts().sort_index(),
    "validation": y_valid.value_counts().sort_index(),
}).fillna(0).astype(int))

## 4. Pipeline y funciones de evaluación

In [0]:
def build_pipeline(config: dict) -> Pipeline:
    # -> Construir un Pipeline con SimpleImputer(median) y el estimador indicado por config
    # ~> Copiar config y extraer metadatos (candidate_name, model_type) que no son hiperparametros
    model_config = dict(config)
    model_config.pop("candidate_name", None)
    model_type = model_config.pop("model_type", "extra_trees")

    # ~> Seleccionar el estimador segun model_type
    if model_type == "extra_trees":
        estimator = ExtraTreesClassifier(**model_config)
    elif model_type == "xgboost":
        estimator = XGBClassifier(**model_config)
    else:
        raise ValueError(f"Familia de modelo no soportada: {model_type}")

    # > Empaquetar imputacion y modelo en un unico Pipeline
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("model", estimator),
        ]
    )


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    # -> Calcular metricas de clasificacion multiclase y latencia por fila
    # ~> Medir tiempo de inferencia
    started = time.perf_counter()
    predictions = model.predict(features)
    latency_ms_per_row = 1000 * (time.perf_counter() - started) / len(features)

    # > Devolver accuracy, precision_macro, recall_macro, f1_macro, f1_weighted y latencia
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_score(labels, predictions, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels, predictions, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "latency_ms_per_row": float(latency_ms_per_row),
    }

## 5. Diseña al menos siete candidatos comparables

In [0]:
# -> Definir al menos siete candidatos comparables con hiperparametros variados
# ~> Todos incluyen random_state y n_jobs=-1 para reproducibilidad y comparacion local
CANDIDATES = [
    # ~> Candidato de referencia: ExtraTrees 300 arboles, max_depth=None, min_samples_leaf=1
    {"candidate_name": "extra_trees_300_leaf_1", "model_type": "extra_trees", "n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    {"candidate_name": "extra_trees_600_leaf_1", "model_type": "extra_trees", "n_estimators": 600, "max_depth": None, "min_samples_leaf": 1, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    {"candidate_name": "extra_trees_300_leaf_2", "model_type": "extra_trees", "n_estimators": 300, "max_depth": None, "min_samples_leaf": 2, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    {"candidate_name": "extra_trees_600_leaf_2", "model_type": "extra_trees", "n_estimators": 600, "max_depth": None, "min_samples_leaf": 2, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    {"candidate_name": "extra_trees_300_leaf_3", "model_type": "extra_trees", "n_estimators": 300, "max_depth": None, "min_samples_leaf": 3, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    {"candidate_name": "extra_trees_600_leaf_3", "model_type": "extra_trees", "n_estimators": 600, "max_depth": None, "min_samples_leaf": 3, "max_features": 1.0, "class_weight": None, "random_state": RANDOM_STATE, "n_jobs": -1},
    # ~> Candidato XGBoost: multi:softprob con tree_method hist
    {
        "candidate_name": "xgboost_hist",
        "model_type": "xgboost",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    },
]

# >> Verificar que hay al menos siete candidatos y al menos uno es XGBoost
assert len(CANDIDATES) >= 7, "Necesitas al menos siete candidatos, incluyendo XGBoost."
assert any(candidate.get("model_type") == "xgboost" for candidate in CANDIDATES)

# >> Mostrar tabla de candidatos
display(pd.DataFrame(CANDIDATES))

## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.joblib` y contrato de inferencia.

In [0]:
# -> Documentar la tarjeta de datos y el registro de riesgos del dataset WineQT
# ~> Tarjeta de datos: origen, proposito, limitaciones conocidas
dataset_card = {
    "source": "UCI Wine Quality / WineQT.csv (copia docente del repositorio)",
    "purpose": "Aprendizaje de MLOps y trazabilidad de clasificacion multiclase.",
    "features": EXPECTED_FEATURES,
    "target": "quality_class",
    "target_definition": f"{TARGET} original 3..8, codificada como {TARGET} - {QUALITY_MIN} para XGBoost",
    "class_mapping": {str(index): int(value) for index, value in enumerate(quality_values)},
    "candidate_families": ["extra_trees", "xgboost"],
    "split": {"train": TRAIN_SIZE, "validation": VALIDATION_SIZE, "test": TEST_SIZE, "random_state": RANDOM_STATE},
    "known_limitations": [
        "La calidad se modela como seis clases; las clases extremas son minoritarias y F1 macro puede ser baja.",
        "El dataset es pequeno y su procedencia no representa toda la produccion vinicola.",
        "No se ha auditado calibracion, deriva, reproducibilidad de laboratorio ni validez comercial.",
    ],
}

# ~> Registro de riesgos: identificar y mitigar riesgos del ciclo de vida
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "Uso de la prediccion como garantia de calidad comercial", "impact": "high", "owner": "product_owner", "mitigation": "Restringir el caso a docencia y requerir validacion de laboratorio."},
        {"id": "R2", "risk": "Sesgo por clases minoritarias o falta de representatividad", "impact": "high", "owner": "data_owner", "mitigation": "Revisar distribucion, procedencia, muestreo y rendimiento por clase."},
        {"id": "R3", "risk": "Diferencia entre preprocessing de training y serving", "impact": "high", "owner": "ml_engineer", "mitigation": "Empaquetar preprocessing y modelo en un unico Pipeline y probar el contrato."},
        {"id": "R4", "risk": "Secretos o datos sensibles en MLflow", "impact": "high", "owner": "team", "mitigation": "No registrar tokens, PII, correos o prompts sensibles."},
    ],
}

# >> Verificar que se cumplen los minimos de documentacion
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:
# -> Crear datasets de MLflow para train y validacion con mlflow.data.from_pandas
# ~> Incluir el target codificado como quality_class, usar DATASET_PATH como source
train_dataset = mlflow.data.from_pandas(
    X_train.assign(quality_class=y_train.to_numpy()), source=DATASET_PATH, name="wine_quality_train"
)
validation_dataset = mlflow.data.from_pandas(
    X_valid.assign(quality_class=y_valid.to_numpy()), source=DATASET_PATH, name="wine_quality_validation"
)

# >> Verificar los nombres de los datasets
print(f"Train dataset: {train_dataset.name}, rows: {len(X_train)}")
print(f"Validation dataset: {validation_dataset.name}, rows: {len(X_valid)}")

### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, Joblib,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    # -> Registrar artefactos del candidato en MLflow
    # ~> Log del modelo con firma, input example y allowlist de skops para XGBoost
    signature = infer_signature(X_valid, model.predict(X_valid))
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=X_valid.head(3),
        skops_trusted_types=SKOPS_TRUSTED_TYPES,
    )
    mlflow.set_tag("candidate.model_uri", model_info.model_uri)

    # ~> Serializar el modelo con joblib y registrar su tamano
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_path = Path(temp_dir)
        joblib_path = temp_path / "model.joblib"
        joblib.dump(model, joblib_path)
        mlflow.log_artifact(str(joblib_path), artifact_path="deployment")
        mlflow.log_metric("model.joblib_size_bytes", joblib_path.stat().st_size)

        # > Guardar matriz de confusion como imagen
        figure, axis = plt.subplots(figsize=(4.5, 4.5))
        ConfusionMatrixDisplay.from_predictions(
            y_valid, predictions, labels=list(range(NUM_CLASSES)), display_labels=quality_values,
            ax=axis, colorbar=False
        )
        axis.set_title(f"Validation · candidate {candidate_index:02d}")
        figure.tight_layout()
        figure_path = temp_path / "validation_confusion_matrix.png"
        figure.savefig(figure_path, dpi=140)
        plt.close(figure)
        mlflow.log_artifact(str(figure_path), artifact_path="evaluation")

    # > Guardar contrato de inferencia con el mapeo de clases
    mlflow.log_dict(
        {
            "ordered_features": EXPECTED_FEATURES,
            "request_key": "instances",
            "target_encoding": f"quality_class = quality - {QUALITY_MIN}",
            "prediction_decoding": f"quality = quality_class + {QUALITY_MIN}",
            "quality_values": quality_values,
        },
        "deployment/inference_contract.json",
    )

    # >> Devolver el model_uri del modelo registrado
    return model_info.model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:
# -> Implementar un bucle de runs, uno por cada candidato
# ~> Registrar identidad, reproducibilidad, datos, gobierno, entrenamiento y artefactos
experiment_id = None
candidate_run_ids = []
candidate_validation_rows = []

for index, config in enumerate(CANDIDATES, start=1):
    run_name = f"{STUDENT_ALIAS}-{BATCH_ID}-{config['candidate_name']}"
    with mlflow.start_run(run_name=run_name) as run:
        # > Guardar experiment_id y run_id
        experiment_id = run.info.experiment_id
        candidate_run_ids.append(run.info.run_id)

        # > Registrar tags de identidad
        mlflow.set_tags(
            {
                "course": "MUIAAP-operacion-modelos",
                "course.week": "01",
                "student.alias": STUDENT_ALIAS,
                "batch.id": BATCH_ID,
                "lifecycle.phase": "candidate",
                "use_case": "didactic-wine-quality-multiclass-classification",
                "risk.tier": "educational-only",
                "candidate.name": config["candidate_name"],
                "candidate.model_type": config["model_type"],
            }
        )

        # > Registrar hiperparametros del candidato y del split
        mlflow.log_params(config)
        mlflow.log_params(
            {
                "split.random_state": RANDOM_STATE,
                "split.train_rows": len(X_train),
                "split.validation_rows": len(X_valid),
                "split.test_rows_reserved": len(X_test),
                "split.train_fraction": TRAIN_SIZE,
                "split.validation_fraction": VALIDATION_SIZE,
                "split.test_fraction": TEST_SIZE,
                "selection.min_validation_f1_macro": MIN_VALIDATION_F1_MACRO,
                "library.sklearn": sklearn.__version__,
                "library.xgboost": xgboost.__version__,
            }
        )

        # > Registrar inputs de train y validacion
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(validation_dataset, context="validation")

        # > Registrar governance: dataset_card, quality_report, risk_register
        mlflow.log_dict(dataset_card, "governance/dataset_card.json")
        mlflow.log_dict(quality_report, "governance/data_quality.json")
        mlflow.log_dict(risk_register, "governance/risk_register.json")

        # > Entrenar el modelo y medir el tiempo de fit
        model = build_pipeline(config)
        fit_started = time.perf_counter()
        model.fit(X_train, y_train)
        mlflow.log_metric("train.fit_seconds", time.perf_counter() - fit_started)

        # > Evaluar en validacion y registrar metricas
        validation_metrics = evaluate_classifier(model, X_valid, y_valid)
        mlflow.log_metrics({f"validation.{k}": v for k, v in validation_metrics.items()})

        # > Imprimir una fila por candidato para comprobar el gate antes de la busqueda
        candidate_validation_rows.append(
            {
                "candidate": config["candidate_name"],
                "model_type": config["model_type"],
                "n_estimators": config.get("n_estimators"),
                "max_depth": config.get("max_depth"),
                "min_samples_leaf": config.get("min_samples_leaf"),
                **{f"validation.{k}": v for k, v in validation_metrics.items()},
            }
        )
        print(
            f"{config['candidate_name']} · {config['model_type']} · "
            f"validation.f1_macro={validation_metrics['f1_macro']:.4f}"
        )

        # > Generar classification_report y predicciones
        predictions = model.predict(X_valid)
        report = classification_report(
            y_valid, predictions, labels=list(range(NUM_CLASSES)),
            target_names=[f"quality={value}" for value in quality_values],
            output_dict=True, zero_division=0,
        )
        mlflow.log_dict(report, "evaluation/validation_classification_report.json")

        # > Registrar artefactos del candidato (modelo, joblib, matriz, contrato)
        model_uri = log_candidate_artifacts(model, predictions, index)

# >> Mostrar tabla comparativa de validacion
training_comparison = pd.DataFrame(candidate_validation_rows).sort_values(
    "validation.f1_macro", ascending=False
)
display(training_comparison)

# >> Verificar que se registraron todos los candidatos
assert len(candidate_run_ids) == len(CANDIDATES)
print(f"Registrados {len(candidate_run_ids)} candidatos en el experimento {experiment_id}.")

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
# -> Buscar y comparar exclusivamente los runs del alias y BATCH_ID actuales
# ~> Filtrar por tres tags: student.alias, batch.id y lifecycle.phase
filter_string = (
    f"tags.student.alias = '{STUDENT_ALIAS}' AND "
    f"tags.batch.id = '{BATCH_ID}' AND "
    "tags.lifecycle.phase = 'candidate'"
)
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=filter_string,
    max_results=100,
)

# > Seleccionar columnas de parametros, metricas de validacion y latencia
metric_columns = [
    "metrics.validation.f1_macro",
    "metrics.validation.f1_weighted",
    "metrics.validation.recall_macro",
    "metrics.validation.precision_macro",
    "metrics.validation.accuracy",
    "metrics.validation.latency_ms_per_row",
    "metrics.model.joblib_size_bytes",
]
columns_to_show = [
    "run_id",
    "tags.mlflow.runName",
    "tags.candidate.model_uri",
    "params.candidate_name",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_leaf",
    "params.model_type",
    "params.learning_rate",
    "params.subsample",
    "params.colsample_bytree",
    "params.class_weight",
    *metric_columns,
]

# > Construir el DataFrame de comparacion
comparison = runs.reindex(columns=columns_to_show).copy()
comparison = comparison.rename(columns={"params.model_type": "model_type"})
for column in metric_columns:
    comparison[column] = pd.to_numeric(comparison[column], errors="coerce")

# >> Mostrar la comparacion ordenada por F1 macro descendente
display(comparison.sort_values("metrics.validation.f1_macro", ascending=False))

## 8. Aplica el gate sin mirar test

Filtra por F1 macro mínimo; ordena por F1 macro DESC, accuracy DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
# -> Aplicar el gate de seleccion sin mirar test
# ~> Filtrar candidatos que superan el F1 macro minimo en validacion
eligible = comparison[
    comparison["metrics.validation.f1_macro"] >= MIN_VALIDATION_F1_MACRO
].copy()

# ~> Si no hay elegibles, fallar sin rebajar el umbral
assert not eligible.empty, (
    "Ningun candidato supera el gate. No rebajes el umbral despues de ver los resultados: "
    "revisa el plan experimental y genera nueva evidencia."
)

# > Ordenar por F1 macro DESC, accuracy DESC y latencia ASC
ranked = eligible.sort_values(
    by=[
        "metrics.validation.f1_macro",
        "metrics.validation.accuracy",
        "metrics.validation.latency_ms_per_row",
    ],
    ascending=[False, False, True],
)

# > Conservar el ganador
winner = ranked.iloc[0]
BEST_RUN_ID = str(winner["run_id"])
BEST_MODEL_URI = str(winner["tags.candidate.model_uri"])

# > Crear selection_report que documenta que test no participo
selection_report = {
    "batch_id": BATCH_ID,
    "candidate_count": int(len(comparison)),
    "eligible_count": int(len(eligible)),
    "gate": {"metric": "validation.f1_macro", "operator": ">=", "value": MIN_VALIDATION_F1_MACRO},
    "ranking": ["validation.f1_macro DESC", "validation.accuracy DESC", "validation.latency_ms_per_row ASC"],
    "winner_run_id": BEST_RUN_ID,
    "winner_model_type": str(winner["model_type"]),
    "test_was_used_for_selection": False,
}

# >> Imprimir el reporte de seleccion
print(json.dumps(selection_report, indent=2))

# >> Mostrar los tres mejores candidatos
display(ranked.head(3))

## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
# -> Cargar el modelo ganador desde MLflow y evaluar test una sola vez
# ~> Cargar el modelo desde el artefacto de MLflow (no de la variable en memoria)
winner_model = mlflow.sklearn.load_model(BEST_MODEL_URI)

# ~> Calcular metricas de test
test_metrics = evaluate_classifier(winner_model, X_test, y_test)

# ~> Crear dataset de test para registrar como input
test_dataset = mlflow.data.from_pandas(
    X_test.assign(quality_class=y_test.to_numpy()), source=DATASET_PATH, name="wine_quality_test"
)

# > Reabrir el run ganador para anadir metricas de test y tags de seleccion
with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_input(test_dataset, context="testing")
    mlflow.log_metrics({f"test.{k}": v for k, v in test_metrics.items()})
    mlflow.set_tags(
        {
            "selection.status": "winner",
            "selection.rule": "f1_macro_gate_then_accuracy_latency",
            "selection.test_used_once": "true",
        }
    )
    mlflow.log_dict(selection_report, "selection/decision.json")

# >> Imprimir el ganador y sus metricas de test
print("Ganador:", BEST_RUN_ID)
print(json.dumps({f"test.{k}": v for k, v in test_metrics.items()}, indent=2))

## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
# -> Registrar el modelo ganador en Unity Catalog Model Registry
# ~> Configurar el registry URI de Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# ~> Obtener catalogo y esquema activos de la sesion
catalog_name, schema_name = spark.sql(
    "SELECT current_catalog(), current_schema()"
).first()

# ~> Saneear el alias para el nombre del modelo (solo a-z, 0-9, _)
safe_alias = re.sub(r"[^a-z0-9_]", "_", STUDENT_ALIAS.lower())
REGISTERED_MODEL_NAME = f"{catalog_name}.{schema_name}.wine_quality_classifier_{safe_alias}"

# > Registrar referencia de rollback (segundo mejor candidato) y ganador
rollback_row = ranked.iloc[1] if len(ranked) > 1 else ranked.iloc[0]
ROLLBACK_MODEL_URI = str(rollback_row["tags.candidate.model_uri"])
rollback_registered = mlflow.register_model(
    model_uri=ROLLBACK_MODEL_URI,
    name=REGISTERED_MODEL_NAME,
)
registered = mlflow.register_model(
    model_uri=BEST_MODEL_URI,
    name=REGISTERED_MODEL_NAME,
)

# > Configurar descripciones y tags del modelo y versiones
registry_client = MlflowClient(registry_uri="databricks-uc")
registry_client.update_registered_model(
    name=REGISTERED_MODEL_NAME,
    description=(
        "Clasificador didactico de Wine Quality de la semana 01. "
        "Ganador elegido mediante gate de F1 macro sobre validacion."
    ),
)
registry_client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=rollback_registered.version,
    description="Referencia para practicar rollback; solo para uso docente.",
)
registry_client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=registered.version,
    description=f"Run ganador {BEST_RUN_ID}; batch {BATCH_ID}.",
)

# > Tags de validacion y uso para ambas versiones
registry_client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME, version=rollback_registered.version,
    key="validation_status", value="rollback_demo_only",
)
registry_client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME, version=rollback_registered.version,
    key="usage_scope", value="educational_only",
)
registry_client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME, version=registered.version,
    key="validation_status", value="passed_didactic_gate",
)
registry_client.set_model_version_tag(
    name=REGISTERED_MODEL_NAME, version=registered.version,
    key="usage_scope", value="educational_only",
)

# > Asignar alias Challenger al ganador y hacer smoke test
registry_client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias="Challenger", version=registered.version,
)
challenger_uri = f"models:/{REGISTERED_MODEL_NAME}@Challenger"
challenger_model = mlflow.sklearn.load_model(challenger_uri)
assert len(challenger_model.predict(X_test.head(3))) == 3

# > Promocionar a Champion
registry_client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Champion", registered.version)
promoted = registry_client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "Champion")
assert str(promoted.version) == str(registered.version)

# > Practicar rollback: asignar Champion a la version de rollback
registry_client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Champion", rollback_registered.version)
rolled_back = registry_client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "Champion")
assert str(rolled_back.version) == str(rollback_registered.version)
rollback_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}@Champion")
assert len(rollback_model.predict(X_test.head(3))) == 3

# > Re-promocionar el ganador a Champion
registry_client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Champion", registered.version)
champion = registry_client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "Champion")
assert str(champion.version) == str(registered.version)

# >> Imprimir el resultado del ciclo Challenger -> Champion -> rollback
print(
    f"Registry verificado: rollback={rollback_registered.version}, "
    f"Champion final={champion.version} (run={champion.run_id})."
)

## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
# -> Smoke test: cargar el modelo desde el alias Champion y predecir tres filas
# ~> El consumidor apunta al alias, no a una version fija
champion_uri = f"models:/{REGISTERED_MODEL_NAME}@Champion"
champion_model = mlflow.sklearn.load_model(champion_uri)

# > Predecir tres filas de test
smoke_predictions = champion_model.predict(X_test.head(3))
assert len(smoke_predictions) == 3

# >> Imprimir el URI y las predicciones (decodificadas a calidad original)
print("URI:", champion_uri)
display(pd.DataFrame({"quality_prediction": smoke_predictions + QUALITY_MIN}))

## 12. API local con el artefacto Joblib ganador

Resuelve el run de `Champion`, descarga `deployment/model.joblib` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
# -> Resolver el run del champion y descargar el joblib del modelo ganador
# ~> Obtener el run_id del champion
source_run_id = champion.run_id or BEST_RUN_ID

# ~> Descargar el artefacto model.joblib desde el run ganador
joblib_uri = f"runs:/{source_run_id}/deployment/model.joblib"
local_joblib_path = mlflow.artifacts.download_artifacts(artifact_uri=joblib_uri)

# > Cargar el modelo con joblib
api_model = joblib.load(local_joblib_path)

# >> Verificar que el modelo puede predecir una fila
test_prediction = api_model.predict(X_test.head(1))
assert len(test_prediction) == 1
print(f"Modelo cargado desde joblib: {local_joblib_path}")
print(f"Prediccion de prueba: quality={test_prediction[0] + QUALITY_MIN}")

### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

# -> Definir PredictionHandler con GET /health y POST /predict
# ~> _send_json: enviar respuesta JSON y actualizar API_STATS en cada respuesta
class PredictionHandler(BaseHTTPRequestHandler):
    def _send_json(self, status: int, payload: dict, started: float) -> None:
        elapsed_ms = 1000 * (time.perf_counter() - started)
        with API_STATS_LOCK:
            API_STATS["requests_total"] += 1
            API_STATS["latencies_ms"].append(elapsed_ms)
            if status < 400:
                API_STATS["requests_success"] += 1
            elif status < 500:
                API_STATS["requests_rejected"] += 1
            else:
                API_STATS["requests_error"] += 1
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:
        started = time.perf_counter()
        if self.path != "/health":
            self._send_json(404, {"error": "not_found"}, started)
            return
        self._send_json(
            200,
            {
                "status": "ok",
                "model": REGISTERED_MODEL_NAME,
                "version": str(champion.version),
                "alias": "Champion",
            },
            started,
        )

    def do_POST(self) -> None:
        started = time.perf_counter()
        if self.path != "/predict":
            self._send_json(404, {"error": "not_found"}, started)
            return
        try:
            # ~> Validar el cuerpo JSON: debe ser un objeto con instances
            content_length = int(self.headers.get("Content-Length", "0"))
            request_payload = json.loads(self.rfile.read(content_length))
            if not isinstance(request_payload, dict):
                raise ValueError("El cuerpo JSON debe ser un objeto.")
            instances = request_payload.get("instances")
            if not isinstance(instances, list) or not instances:
                raise ValueError("'instances' debe ser una lista no vacia.")
            if not all(isinstance(instance, dict) for instance in instances):
                raise ValueError("Cada elemento de 'instances' debe ser un objeto.")

            # > Validar contrato: columnas exactas
            frame = pd.DataFrame(instances)
            missing = sorted(set(EXPECTED_FEATURES) - set(frame.columns))
            extra = sorted(set(frame.columns) - set(EXPECTED_FEATURES))
            if missing or extra:
                raise ValueError(f"Contrato invalido; missing={missing}, extra={extra}")
            frame = frame[EXPECTED_FEATURES]

            # > Predecir y devolver clase, etiquetas y probabilidades
            predictions = api_model.predict(frame)
            probabilities = api_model.predict_proba(frame)
            self._send_json(
                200,
                {
                    "predictions": [int(value + QUALITY_MIN) for value in predictions],
                    "class_labels": [int(value + QUALITY_MIN) for value in api_model.classes_],
                    "probabilities": [[float(value) for value in row] for row in probabilities],
                    "model_version": str(champion.version),
                },
                started,
            )
        except (ValueError, TypeError, json.JSONDecodeError) as error:
            self._send_json(400, {"error": "invalid_request", "detail": str(error)}, started)
        except Exception:
            self._send_json(500, {"error": "internal_error"}, started)

    def log_message(self, format: str, *args) -> None:
        return

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:
# -> Arrancar el servidor HTTP en 127.0.0.1 con puerto automatico
# ~> Crear el servidor con ThreadingHTTPServer y puerto 0 (auto)
api_server = ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
API_PORT = api_server.server_address[1]

# ~> Lanzar el servidor en un hilo daemon
api_thread = threading.Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()

# > Definir stop_api() para apagar el servidor de forma idempotente
API_STOPPED = False

def stop_api() -> None:
    global API_STOPPED
    if not API_STOPPED:
        api_server.shutdown()
        api_server.server_close()
        api_thread.join(timeout=5)
        API_STOPPED = True

# >> Imprimir la URL de la API local
print(f"API local activa en http://127.0.0.1:{API_PORT}")

## 13. Pruebas de contrato y de error

In [0]:
# -> Probar el contrato de la API: health, prediccion valida y errores 400
# ~> Definir call_json para hacer peticiones HTTP y medir latencia
def call_json(method: str, path: str, payload: object | None = None) -> tuple[int, dict, float]:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"http://127.0.0.1:{API_PORT}{path}",
        data=body,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    started = time.perf_counter()
    try:
        with urlopen(request, timeout=10) as response:
            status = response.status
            result = json.loads(response.read())
    except HTTPError as error:
        status = error.code
        result = json.loads(error.read())
    elapsed_ms = 1000 * (time.perf_counter() - started)
    return status, result, elapsed_ms


# > Ejecutar pruebas con try/finally para apagar siempre la API
try:
    # ~> GET /health -> 200
    health_status, health_body, health_ms = call_json("GET", "/health")
    # ~> POST valido -> 200
    example_instance = json.loads(X_test.head(1).to_json(orient="records"))[0]
    predict_status, predict_body, predict_ms = call_json(
        "POST", "/predict", {"instances": [example_instance]}
    )
    # ~> POST con solo alcohol -> 400 (columnas incompletas)
    invalid_status, invalid_body, invalid_ms = call_json(
        "POST", "/predict", {"instances": [{"alcohol": 10.5}]}
    )
    # ~> POST con JSON no-object -> 400
    non_object_status, non_object_body, non_object_ms = call_json(
        "POST", "/predict", [example_instance]
    )

    # > Verificar los resultados esperados
    assert health_status == 200 and health_body["status"] == "ok"
    assert predict_status == 200 and len(predict_body["predictions"]) == 1
    assert invalid_status == 400 and invalid_body["error"] == "invalid_request"
    assert non_object_status == 400 and non_object_body["error"] == "invalid_request"
finally:
    stop_api()

# >> Imprimir los resultados de las pruebas
print("health:", health_status, health_body)
print("predict:", predict_status, predict_body)
print("missing columns:", invalid_status, invalid_body)
print("non-object JSON:", non_object_status, non_object_body)

## 14. Registra un run separado para la evidencia del despliegue

In [0]:
# -> Registrar un run separado para la evidencia del despliegue
# ~> Crear el diccionario de evidencia con los resultados de las pruebas
deployment_evidence = {
    "mode": "local_notebook_http",
    "public_endpoint": False,
    "persistent": False,
    "model_registry_name": REGISTERED_MODEL_NAME,
    "model_version": str(champion.version),
    "model_alias": "Champion",
    "source_run_id": source_run_id,
    "tests": {
        "health": {"status": health_status},
        "valid_prediction": {"status": predict_status, "response_keys": sorted(predict_body)},
        "invalid_contract": {"status": invalid_status, "error": invalid_body.get("error")},
        "non_object_json": {"status": non_object_status, "error": non_object_body.get("error")},
    },
    "limitations": [
        "Solo accesible desde el driver del notebook.",
        "Se detiene al terminar la sesion; no tiene SLA ni autenticacion.",
        "No es apta para trafico o datos reales.",
    ],
}

# > Crear un run de despliegue con tags, params y metricas agregadas
with mlflow.start_run(run_name=f"{STUDENT_ALIAS}-{BATCH_ID}-local-deployment") as deployment_run:
    mlflow.set_tags(
        {
            "course.week": "01",
            "student.alias": STUDENT_ALIAS,
            "batch.id": BATCH_ID,
            "lifecycle.phase": "deployment-test",
            "deployment.mode": "local-notebook",
            "deployment.public": "false",
            "model.source_run_id": source_run_id,
        }
    )
    mlflow.log_params(
        {
            "registered_model": REGISTERED_MODEL_NAME,
            "model_version": str(champion.version),
            "model_alias": "Champion",
        }
    )
    mlflow.log_metrics(
        {
            "api.requests_total": API_STATS["requests_total"],
            "api.requests_success": API_STATS["requests_success"],
            "api.requests_rejected": API_STATS["requests_rejected"],
            "api.requests_error": API_STATS["requests_error"],
            "api.latency_mean_ms": float(np.mean(API_STATS["latencies_ms"])),
            "api.latency_max_ms": float(np.max(API_STATS["latencies_ms"])),
            "api.health_latency_ms": health_ms,
            "api.predict_latency_ms": predict_ms,
            "api.invalid_latency_ms": invalid_ms,
        }
    )
    mlflow.log_dict(deployment_evidence, "deployment/evidence.json")
    DEPLOYMENT_RUN_ID = deployment_run.info.run_id
    # > Registrar la ficha reproducible del lote junto con el run de despliegue
    project_record = {
        "project": {"name": "clasificacion_wine_quality_didactica", "team_alias": STUDENT_ALIAS},
        "data": {"source": DATASET_PATH, "rows": len(data), "personal_data": False},
        "evidence": {
            "experiment_id": experiment_id,
            "batch_id": BATCH_ID,
            "candidate_run_ids": candidate_run_ids,
            "selection_gate": selection_report,
            "winner": {
                "run_id": BEST_RUN_ID,
                "validation_f1_macro": float(winner["metrics.validation.f1_macro"]),
                "validation_recall_macro": float(winner["metrics.validation.recall_macro"]),
                "test_f1_macro": test_metrics["f1_macro"],
                "test_recall_macro": test_metrics["recall_macro"],
            },
        },
        "registry": {
            "registered_model": REGISTERED_MODEL_NAME,
            "rollback_version": str(rollback_registered.version),
            "champion_version": str(champion.version),
            "alias": "Champion",
        },
        "deployment": {
            "run_id": DEPLOYMENT_RUN_ID,
            "tests": deployment_evidence["tests"],
            "api_stopped": API_STOPPED,
        },
        "risks": risk_register["risks"],
    }
    mlflow.log_dict(project_record, "governance/s01_project_record.json")

# >> Imprimir el run de despliegue
print("Run de despliegue:", DEPLOYMENT_RUN_ID)

## 15. Apaga la API y entrega el resumen del ciclo

In [0]:
# -> Apagar la API y entregar el resumen del ciclo completo
# ~> Detener el servidor y verificar que el hilo termino
stop_api()
assert not api_thread.is_alive(), "El servidor no se apago correctamente."

# > Construir el resumen final del ciclo
summary = {
    "experiment_id": experiment_id,
    "batch_id": BATCH_ID,
    "candidate_runs": len(candidate_run_ids),
    "winner_run_id": BEST_RUN_ID,
    "registered_model": REGISTERED_MODEL_NAME,
    "registered_version": str(champion.version),
    "alias": "Champion",
    "deployment_run_id": DEPLOYMENT_RUN_ID,
    "api_stopped": API_STOPPED,
}

# >> Imprimir el resumen final
print(json.dumps(summary, indent=2))